# 03 - Matching, Validation, and Event Selection

Goal: study the ingredients that drive neutrino event selection by checking PID, primary labeling, vertex quality, and matching-based diagnostics on a realistic sample.

Notebook 2 covered SPINE object inspection. This notebook introduces truth matching and validation before we turn the objects into a specific analysis in Notebook 4.

## What you should already know

This notebook reuses the same file-loading setup as Notebook 2, without repeating the full YAML.

It defaults to the `nd-lar_lbnf` sample rather than `2x2_numi` because this detector/sample pair is a better place to discuss neutrino event-selection ingredients such as PID, primaries, and vertex quality.

The new concepts here are:

- truth-to-reco matching;
- overlap thresholds;
- confusion matrices;
- simple validation plots that tell you where to look next.

## Step 1: import the tools for this notebook

This first code cell does not touch any data yet. It only imports the Python modules we will use later.

If an import fails, stop here. There is no point debugging later cells until this environment cell runs cleanly.

In [ ]:

# Standard Python/path tools for working with local files.
from pathlib import Path

# Analysis utilities used throughout the tutorials.
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

# The Driver is the main SPINE entry point for reading one event at a time.
from spine.driver import Driver

# For richer config composition than this tutorial's minimal YAML readback.
from spine.config import load_config_file


## Step 2: choose the input file

This second code cell answers one question: which SPINE HDF5 file should the rest of the notebook read?

It looks for the shared workshop area first, then for the repository-local tutorial assets. It also defines the detector name and sample tag that control which file gets opened.

In [ ]:

# Prefer the shared DUNE path used in the workshop environment.
# Fall back to the repository-local tutorial assets when needed.
# We include both relative spellings because notebook kernels do not always
# start in the notebook directory.
DATA_ROOT_CANDIDATES = [
    Path("/exp/dune/data/users/drielsma/npc-ddas"),
    Path("../assets"),
    Path("tutorials/assets"),
]

# Pick the first location that actually exists on disk.
DATA_ROOT = next((path for path in DATA_ROOT_CANDIDATES if path.exists()), None)
if DATA_ROOT is None:
    raise RuntimeError("Could not find a workshop data directory.")

# The reco HDF5 files live under reco/ inside the chosen data root.
HDF5_DATA_DIR = DATA_ROOT / "reco"

# Edit these values to switch samples.
# The expected layout is:
#   reco/DETECTOR/SAMPLE_NAME_spine.h5
DETECTOR = "nd-lar"
SAMPLE_NAME = "nd-lar_lbnf"

# GEOMETRY tells SPINE which detector geometry description to use.
# For this workshop we use the detector name directly.
GEOMETRY = DETECTOR
HDF5_FILE_NAME = f"{SAMPLE_NAME}_spine.h5"

# Build the final path to the reconstructed SPINE HDF5 file.
DATA_PATH = HDF5_DATA_DIR / DETECTOR / HDF5_FILE_NAME

print(f"Using data root: {DATA_ROOT}")
print(f"Reco file: {DATA_PATH}")


## Step 3: build the SPINE driver from YAML

Now that the notebook knows which file to read, it loads the tutorial YAML config, injects the concrete file path, and creates a `Driver` object.

The geometry override in this step is important: it tells SPINE which detector geometry description to attach when the chosen sample needs one.

In [ ]:

# As with the data paths, the notebook working directory can vary.
# These two candidates point to the same tutorial config from different cwd values.
CONFIG_CANDIDATES = [
    Path("../config/read_spine_hdf5.yaml"),
    Path("tutorials/config/read_spine_hdf5.yaml"),
]
CONFIG_PATH = next((path for path in CONFIG_CANDIDATES if path.exists()), None)

# Replace the DATA_PATH placeholder in the YAML template with the file we chose above.
DATA_PATH = str(DATA_PATH)
if CONFIG_PATH is None:
    raise RuntimeError("Could not find tutorials/config/read_spine_hdf5.yaml")

# Read the YAML as text first so we can substitute the concrete file path.
cfg_text = CONFIG_PATH.read_text().replace("DATA_PATH", DATA_PATH)

# Convert the YAML text into a normal Python dictionary.
cfg = yaml.safe_load(cfg_text)

# Some detector samples need an explicit geometry block so downstream code knows
# which detector layout and coordinate conventions to use.
if GEOMETRY:
    cfg["geo"] = {"detector": GEOMETRY}

# Create the SPINE driver. From this point on, driver.process(entry=...) reads events.
driver = Driver(cfg)
print(f"Opened {DATA_PATH}")
print(f"Entries: {len(driver)}")
if GEOMETRY:
    print(f"Detector geometry: {GEOMETRY}")


## Config loading note: `safe_load` vs `load_config_file`

The setup cell above uses `yaml.safe_load` because this tutorial config is deliberately tiny: it reads one YAML template, replaces `DATA_PATH`, and creates a normal Python dictionary.

For real SPINE configuration work, prefer `spine.config.load_config_file`. That loader understands SPINE's config composition features, including `include`, `!include`, `!path`, `!download`, `override`, and removal operations. It is the better tool when you want to start from a production config and add or modify blocks in a notebook.

For example:

```python
cfg = load_config_file("/path/to/base_config.yaml")
cfg["ana"] = {"save": save_cfg}
driver = Driver(cfg)
```

Use `help(load_config_file)` when you want to explore its options interactively.

In [ ]:
from sklearn.metrics import confusion_matrix

N_ENTRIES = min(len(driver), 100)
MATCH_THRESHOLD = 0.1
print(f"Validating {N_ENTRIES} entries with overlap threshold {MATCH_THRESHOLD}")

## Sanity-check one event first

Before aggregating many events, inspect one event and ask:

1. Are there matched particle pairs at all?
2. Are there matched interaction pairs at all?
3. What does the overlap value look like for this event?

In [ ]:
ENTRY = 0
example = driver.process(entry=ENTRY)

particle_pairs = example.get("particle_matches_t2r", [])
particle_overlaps = example.get("particle_matches_t2r_overlap", np.ones(len(particle_pairs)))
interaction_pairs = example.get("interaction_matches_t2r", [])
interaction_overlaps = example.get("interaction_matches_t2r_overlap", np.ones(len(interaction_pairs)))

In [ ]:
pd.DataFrame(
    {
        "collection": ["particle_matches_t2r", "interaction_matches_t2r"],
        "count": [len(particle_pairs), len(interaction_pairs)],
        "max_overlap": [
            float(np.max(particle_overlaps)) if len(particle_overlaps) else np.nan,
            float(np.max(interaction_overlaps)) if len(interaction_overlaps) else np.nan,
        ],
    }
)

## Build validation tables

The next cell converts matched objects into four ordinary tables:

- one row per matched particle pair;
- one row per matched primary-label comparison;
- one row per matched interaction pair for event-class validation;
- one row per matched interaction pair for vertex validation.

Once the tables exist, the rest of the notebook is mostly pandas and plotting.

In [ ]:
particle_rows = []
primary_rows = []
interaction_rows = []
vertex_rows = []


def interaction_class(interaction):
    # In this tutorial we collapse neutrino interactions into three categories:
    #   CC numu: at least one primary muon
    #   CC nue:  at least one primary electron
    #   NC nu:   no primary lepton
    primary_pids = {
        getattr(p, "pid", -1)
        for p in getattr(interaction, "particles", [])
        if getattr(p, "is_primary", False)
    }
    if 2 in primary_pids:
        return "CC numu"
    if 1 in primary_pids:
        return "CC nue"
    return "NC nu"

for entry in range(N_ENTRIES):
    data = driver.process(entry=entry)

    pairs = data.get("particle_matches_t2r", [])
    overlaps = data.get("particle_matches_t2r_overlap", np.ones(len(pairs)))
    for i, (truth_p, reco_p) in enumerate(pairs):
        overlap = overlaps[i]
        if overlap < MATCH_THRESHOLD:
            continue
        if getattr(truth_p, "pid", -1) < 0 or getattr(reco_p, "pid", -1) < 0:
            continue
        particle_rows.append({
            "entry": entry,
            "overlap": overlap,
            "true_pid": truth_p.pid,
            "reco_pid": reco_p.pid,
            "true_primary": bool(getattr(truth_p, "is_primary", False)),
            "reco_primary": bool(getattr(reco_p, "is_primary", False)),
            "true_size": getattr(truth_p, "size", np.nan),
            "reco_size": getattr(reco_p, "size", np.nan),
            "true_nu_id": getattr(truth_p, "nu_id", -1),
        })
        primary_rows.append({
            "entry": entry,
            "truth": bool(getattr(truth_p, "is_primary", False)),
            "reco": bool(getattr(reco_p, "is_primary", False)),
            "overlap": overlap,
            "true_nu_id": getattr(truth_p, "nu_id", -1),
        })

    ia_pairs = data.get("interaction_matches_t2r", [])
    ia_overlaps = data.get("interaction_matches_t2r_overlap", np.ones(len(ia_pairs)))
    for i, (truth_ia, reco_ia) in enumerate(ia_pairs):
        overlap = ia_overlaps[i]
        if overlap < MATCH_THRESHOLD:
            continue
        interaction_rows.append({
            "entry": entry,
            "truth_interaction_id": getattr(truth_ia, "id", -1),
            "reco_interaction_id": getattr(reco_ia, "id", -1),
            "overlap": overlap,
            "true_nu_id": getattr(truth_ia, "nu_id", -1),
            "true_class": interaction_class(truth_ia),
            "reco_class": interaction_class(reco_ia),
        })
        if not hasattr(truth_ia, "vertex") or not hasattr(reco_ia, "vertex"):
            continue
        vertex_rows.append({
            "entry": entry,
            "truth_interaction_id": getattr(truth_ia, "id", -1),
            "reco_interaction_id": getattr(reco_ia, "id", -1),
            "overlap": overlap,
            "true_nu_id": getattr(truth_ia, "nu_id", -1),
            "vertex_distance_cm": float(np.linalg.norm(truth_ia.vertex - reco_ia.vertex)),
        })

particles = pd.DataFrame(particle_rows)
primaries = pd.DataFrame(primary_rows)
interactions = pd.DataFrame(interaction_rows)
vertices = pd.DataFrame(vertex_rows)

display(particles.head())
display(interactions.head())
display(vertices.head())

## Ask simple questions before plotting

Before reading the confusion matrix, check that you understand the table columns:

1. Which columns come from truth?
2. Which columns come from reconstruction?
3. Which columns are bookkeeping or metadata?
4. Which filters below are removing unmatched or non-neutrino entries?

In [ ]:
from spine.constants import PID_LABELS

PID_LABELS = [PID_LABELS[i] for i in range(5)]
valid_pid = particles.query("0 <= true_pid <= 4 and 0 <= reco_pid <= 4")

cm_counts = confusion_matrix(valid_pid["true_pid"], valid_pid["reco_pid"], labels=[0, 1, 2, 3, 4])
cm_norm = confusion_matrix(
    valid_pid["true_pid"],
    valid_pid["reco_pid"],
    labels=[0, 1, 2, 3, 4],
    normalize="true",
)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm_norm, vmin=0, vmax=1, cmap="Blues")
plt.colorbar(im, ax=ax, label="row-normalized fraction")
ax.set_xticks(range(5), PID_LABELS, rotation=45, ha="right")
ax.set_yticks(range(5), PID_LABELS)
ax.set_xlabel("reco PID")
ax.set_ylabel("true PID")
for i in range(5):
    for j in range(5):
        ax.text(j, i, f"{cm_norm[i, j]:.2f}\n({cm_counts[i, j]})", ha="center", va="center", fontsize=8)
plt.tight_layout()

## Primary-particle validation

Here we restrict to neutrino-associated truth particles as a very simple slice of the dataset. That is what the `true_nu_id > -1` cut is doing.

In [ ]:
mpv_primary = primaries[primaries["true_nu_id"] > -1]
primary_counts = confusion_matrix(mpv_primary["truth"], mpv_primary["reco"], labels=[False, True])
primary_norm = confusion_matrix(mpv_primary["truth"], mpv_primary["reco"], labels=[False, True], normalize="true")

fig, ax = plt.subplots(figsize=(4, 4))
im = ax.imshow(primary_norm, vmin=0, vmax=1, cmap="Greens")
plt.colorbar(im, ax=ax, label="row-normalized fraction")
ax.set_xticks([0, 1], ["non-primary", "primary"], rotation=30, ha="right")
ax.set_yticks([0, 1], ["non-primary", "primary"])
ax.set_xlabel("reco")
ax.set_ylabel("truth")
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{primary_norm[i, j]:.2f}\n({primary_counts[i, j]})", ha="center", va="center")
plt.tight_layout()

## Vertex-resolution diagnostic

This is intentionally simple: for matched truth and reco interactions, compute the distance between the two vertices and look at the tail.

In [ ]:
mpv_vertices = vertices[vertices["true_nu_id"] > -1]
fig, ax = plt.subplots(figsize=(6, 3))
mpv_vertices["vertex_distance_cm"].hist(ax=ax, bins=30)
ax.set_xlabel("truth-reco vertex distance [cm]")
ax.set_ylabel("matched interactions")
ax.grid(alpha=0.3)

bad_vertices = mpv_vertices.sort_values("vertex_distance_cm", ascending=False).head(10)
bad_vertices

## Neutrino event-class confusion matrix

For event selection, a very common first question is whether SPINE gets the broad interaction class right.

Here we collapse matched neutrino interactions into three bins:

- `CC numu`: one or more primary muons;
- `CC nue`: one or more primary electrons;
- `NC nu`: no primary lepton.

This is a compact diagnostic to follow up in the event display once you have already looked at the vertex behavior.

In [ ]:
EVENT_CLASS_LABELS = ["CC numu", "CC nue", "NC nu"]

nu_interactions = interactions[interactions["true_nu_id"] > -1]
event_counts = confusion_matrix(
    nu_interactions["true_class"],
    nu_interactions["reco_class"],
    labels=EVENT_CLASS_LABELS,
)
event_norm = confusion_matrix(
    nu_interactions["true_class"],
    nu_interactions["reco_class"],
    labels=EVENT_CLASS_LABELS,
    normalize="true",
)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(event_norm, vmin=0, vmax=1, cmap="Purples")
plt.colorbar(im, ax=ax, label="row-normalized fraction")
ax.set_xticks(range(3), EVENT_CLASS_LABELS, rotation=30, ha="right")
ax.set_yticks(range(3), EVENT_CLASS_LABELS)
ax.set_xlabel("reco event class")
ax.set_ylabel("truth event class")
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{event_norm[i, j]:.2f}\n({event_counts[i, j]})", ha="center", va="center", fontsize=9)
plt.tight_layout()

## Live exercise

Pick one bad vertex or one PID confusion mode and send that event back to Spinal Tap. Then decide what kind of failure you are looking at:

- segmentation;
- fragmentation;
- interaction clustering;
- PID;
- primary labeling;
- vertexing.

The important shift here is from counting mistakes to diagnosing mistakes.

## Offline extensions

- Split PID confusion by particle length, deposited charge, containment, or detector module boundary.
- Build efficiency and purity curves for the selection in Notebook 4.
- Compare validation metrics before and after changing a production modifier or post-processing threshold.
- Turn one recurring failure mode into a short debugging note with event IDs and screenshots.

## Real-analysis checklist

- Record the exact SPINE and `spine-prod` versions used to make the file.
- Keep the inference config, weight path or tag, detector geometry, and input file provenance.
- Validate object definitions before optimizing cuts.
- Keep event-display debugging tied to table rows; screenshots without entry IDs are not reproducible.